NOTA: il codice qui presente, data la complessità delle librerie è stato prima generato tramite Gemini Pro e successivamente, per quanto possibile, revisionato.

-----------------------------------------------------------------

Alcuni pacchetti non sono presenti di default, devono quindi essere installati

In [3]:
!pip install ase dscribe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 777.4/777.4 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 13.5 MB/s eta 0:00:00


In [4]:
import numpy as np
#from ase.io import read
import ase.io
from dscribe.descriptors import SOAP
import time

In [5]:
!pwd

/content


Vogliamo prendere i vari file di dati che abbiamo a disposizione ed estrarne la struttura fisica tramite ASE.

Abbiamo una dinamica molecolare particolarmente lunga, quindi quello che proviamo a fare per priamo cosa è leggere i vari file e caricarli in un unico vettore. Se questo fallisce per questioni di fattibilità computazionale lavoreremo con 9 file separatamente e successivamente andremo ad unirli se possibile

# ASE (all in one)


I file sono stati caricati su google drive, per riusare questo notebook i path vanno cambiati

In [10]:
#Elenco dei file da leggere
file_list = (
    #"/content/drive/MyDrive/Università/Tesi/TestData/tantala_md_temp3000.out",
    "/content/drive/MyDrive/Università/Tesi/TestData/tantala_md_temp3000.out_2",
    "/content/drive/MyDrive/Università/Tesi/TestData/tantala_md_temp3000.out_3",
    "/content/drive/MyDrive/Università/Tesi/TestData/tantala_md_temp3000.out_4",
    "/content/drive/MyDrive/Università/Tesi/TestData/tantala_md_temp3000.out_5",
    "/content/drive/MyDrive/Università/Tesi/TestData/tantala_md_temp3000.out_6",
    "/content/drive/MyDrive/Università/Tesi/TestData/tantala_md_temp3000.out_7",
    "/content/drive/MyDrive/Università/Tesi/TestData/tantala_md_temp3000.out_8",
    "/content/drive/MyDrive/Università/Tesi/TestData/tantala_md_temp3000.out_9"
)

Non è stato possibile leggere il primo file in quanto ritornava un:

AssertionError: ((2, 0), 1)

Interpretato by Gemini come segue:

The AssertionError: ((2, 0), 1) in your code indicates that there's an issue with the format of your Quantum ESPRESSO output file, specifically a mismatch between the number of eigenvalues and k-points that ASE's reader expects. This usually means the .out file is either incomplete or corrupted. Please check the file /content/drive/MyDrive/Università/Tesi/TestData/tantala_md_temp3000.out to ensure it's a valid and complete Quantum ESPRESSO output.


In [8]:
traiettoria_completa = []

In [11]:
# 2. Loop di lettura
start_time_reading = time.time()

for i, file_name in enumerate(file_list):
    print(f"Lettura del file di Quantum ESPRESSO: {file_name} in corso ...")

    # Leggiamo tutti i frame del file corrente
    # index=":" dice ad ASE di leggere TUTTI gli step di dinamica, non solo l'ultimo
    # Restituisce una lista di oggetti "Atoms"
    frames = ase.io.read(file_name, index=":", format="espresso-out")


    #Printiamo alcune informazioni sul file letto
    num_frames = len(frames)
    num_atomi = len(frames[0])
    print(f"Letti {num_frames} frame di dinamica molecolare.")
    print(f"Ogni frame contiene {num_atomi} atomi.")


    # NOTA SUI RESTART: Spesso Quantum ESPRESSO, quando riparte,
    # riscrive il frame iniziale che è identico all'ultimo del file precedente.
    if i > 0:
        # Controlliamo se il primo frame del nuovo file è uguale all'ultimo già salvato
        # Se sì, lo scartiamo per non avere duplicati nella statistica
        traiettoria_completa.extend(frames[1:])
    else:
        traiettoria_completa.extend(frames)

print(f"Totale frame accumulati: {len(traiettoria_completa)}")

end_time_reading = time.time()


#Procediamo anche a salvare i la traiettoria letta in modo che non sia necessario procedere con il calcolo tutte le volte che essa deve essere utilizzata

# Salva l'intera lista di atomi in un unico file
ase.io.write('/content/drive/MyDrive/Università/Tesi/TestData/traiettoria_unita.traj', traiettoria_completa)
# Oppure in formato testo leggibile:
# ase.io.write('traiettoria_unita.extxyz', traiettoria_completa)

Lettura del file di Quantum ESPRESSO: /content/drive/MyDrive/Università/Tesi/TestData/tantala_md_temp3000.out_2 in corso ...
Letti 2070 frame di dinamica molecolare.
Ogni frame contiene 126 atomi.
Lettura del file di Quantum ESPRESSO: /content/drive/MyDrive/Università/Tesi/TestData/tantala_md_temp3000.out_3 in corso ...
Letti 134 frame di dinamica molecolare.
Ogni frame contiene 126 atomi.
Lettura del file di Quantum ESPRESSO: /content/drive/MyDrive/Università/Tesi/TestData/tantala_md_temp3000.out_4 in corso ...
Letti 400 frame di dinamica molecolare.
Ogni frame contiene 126 atomi.
Lettura del file di Quantum ESPRESSO: /content/drive/MyDrive/Università/Tesi/TestData/tantala_md_temp3000.out_5 in corso ...
Letti 400 frame di dinamica molecolare.
Ogni frame contiene 126 atomi.
Lettura del file di Quantum ESPRESSO: /content/drive/MyDrive/Università/Tesi/TestData/tantala_md_temp3000.out_6 in corso ...
Letti 400 frame di dinamica molecolare.
Ogni frame contiene 126 atomi.
Lettura del file di

In [12]:
#Diamo una stima del tempo impiegato per la lettura
print(f"Calcolo terminato in {end_time_reading - start_time_reading:.1f} secondi.")

Calcolo terminato in 24.7 secondi.


In [ ]:
# Lettura della traiettoria

# traiettoria_caricata = ase.io.read('traiettoria_unita.traj', index=':')

# ASE (step by step)

# SOAP

Usiamo adesso i dati processati da ASE per produrre il SOAP

In [14]:
# 3. Ora puoi passare la lista unica a DScribe
start_time_soap = time.time()

# Configurare il Descrittore SOAP
# i parametri qui inseriti che si riferiscono alla "risoluzione" del risultato sono stati inseriti da Gemini e possono essere facilmente modficati
soap = SOAP(
    species=["Ta", "O"], # Gli elementi chimici presenti nel tuo sistema
    periodic=True,       # Essenziale: è una cella periodica di bulk!
    r_cut=5.0,            # Raggio di taglio in Angstrom (5.0 è un buon punto di partenza)
    n_max=8,              # Numero di funzioni di base radiali (risoluzione radiale)
    l_max=6               # Grado massimo delle armoniche sferiche (risoluzione angolare)
    )

# Creiamo i descrittori per l'intera traiettoria.
# n_jobs=-1 usa tutti i core della tua CPU per parallelizzare il calcolo.
features = soap.create(traiettoria_completa, n_jobs=-1)

print(f"Matrice SOAP finale: {features.shape}")

end_time_soap = time.time()


#Procediamo anche a salvare i il vettore soap in modo che non sia necessario procedere con il calcolo tutte le volte che esso deve essere utilizzato

# 'features' è l'output di soap.create
np.save('/content/drive/MyDrive/Università/Tesi/TestData/features_soap.npy', features)
#nota: questo file potrebbe essere paricolarmente pesante

Matrice SOAP finale: (4597, 126, 952)


In [15]:
#Diamo una stima del tempo impiegato per la lettura
print(f"Calcolo terminato in {end_time_soap - start_time_soap:.1f} secondi.")

Calcolo terminato in 181.2 secondi.


In [16]:
# Convertiamo in un array NumPy per facilitare il lavoro con il Machine Learning
soap_array = np.array(features)

In [ ]:
# Lettura delle feature

# features_caricate = np.load('features_soap.npy')

# EXTRA

In [17]:
# ==========================================
# EXTRA: Estrarre Energie e Forze (Le tue "Label" Y)
# ==========================================
# Se stai addestrando un modello, avrai bisogno di associare ogni SOAP alla sua energia/forza
energie = np.array([atoms.get_potential_energy() for atoms in traiettoria_completa])
forze = np.array([atoms.get_forces() for atoms in traiettoria_completa])

print(f"Formato array Energie: {energie.shape}")
print(f"Formato array Forze: {forze.shape}")

Formato array Energie: (4597,)
Formato array Forze: (4597, 126, 3)
